#### **1. IMPORTING LIBRARIES**

In [6]:
import os
import cv2
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split


#### **2. SCANNINIG DIRECTORIES & DATA LOADING**

In [7]:
print("Executing Step 1: Scanning folders and ingesting raw images...")
data_dir = 'PetImages'
target_size = (64, 64)

images_accumulator = []
labels_accumulator = []
metadata_records = []

categories = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]
print(f"Detected Classes: {categories}")

for category in categories:
    category_path = os.path.join(data_dir, category)

    for file_name in os.listdir(category_path):
        file_path = os.path.join(category_path, file_name)

        # Read the image file using OpenCV
        img = cv2.imread(file_path)
        if img is None:
            continue

        orig_h, orig_w, _ = img.shape

        # Step 2: Spatial Standardization (Resizing)
        resized_img = cv2.resize(img, target_size)

        # Step 3: Numerical Normalization (Scaling between 0.0 and 1.0)
        normalized_img = resized_img.astype('float32') / 255.0

        # Store processed matrices and text labels
        images_accumulator.append(normalized_img)
        labels_accumulator.append(category)

        # Log properties for the metadata tracking file
        metadata_records.append({
            'File_Name': file_name,
            'Original_Width': orig_w,
            'Original_Height': orig_h,
            'Assigned_Label': category
        })

# Convert processed lists into standard machine learning NumPy arrays
X = np.array(images_accumulator)
y = np.array(labels_accumulator)

Executing Step 1: Scanning folders and ingesting raw images...
Detected Classes: ['Cat', 'Dog']



#### **2. STRATIFIED DATA SPLITTING**

In [8]:
print("Executing Step 4: Stratifying dataset into Train/Validation split partitions...")

indices = np.arange(len(X))

# Perform 80% training / 20% validation split
idx_train, idx_val, _, _ = train_test_split(
    indices, y, test_size=0.2, random_state=1, stratify=y
)

# Allocate partition split tags directly back to metadata dictionary
for idx in idx_train:
    metadata_records[idx]['Data_Split'] = 'Train'
for idx in idx_val:
    metadata_records[idx]['Data_Split'] = 'Validation'

# Extract finalized tensor sets for modeling
X_train, X_val = X[idx_train], X[idx_val]
y_train, y_val = y[idx_train], y[idx_val]

Executing Step 4: Stratifying dataset into Train/Validation split partitions...



#### **3. METADATA EXPORT LOGGING**

In [10]:
print("Executing Step 5: Exporting processing records to 'metadata_log.csv'...")
df_meta = pd.DataFrame(metadata_records)
df_meta.to_csv('metadata_log.csv', index=False)

print("\n" + "="*45)
print("      IMAGE CLASSIFICATION PIPELINE STATS    ")
print("="*45)
print(f"Training Images Shape Vector:   {X_train.shape}")
print(f"Validation Images Shape Vector: {X_val.shape}")
print(f"Training Labels Array Count:    {y_train.shape[0]} items")
print(f"Validation Labels Array Count:  {y_val.shape[0]} items")
print("="*45 + "\nPipeline Success: Preprocessed features locked.")

Executing Step 5: Exporting processing records to 'metadata_log.csv'...

      IMAGE CLASSIFICATION PIPELINE STATS    
Training Images Shape Vector:   (19997, 64, 64, 3)
Validation Images Shape Vector: (5000, 64, 64, 3)
Training Labels Array Count:    19997 items
Validation Labels Array Count:  5000 items
Pipeline Success: Preprocessed features locked.
